# Variational Monte Carlo in continuous space
## Stage 1 — Noninteracting trapped spin-1/2 fermions in 2D


We use $\hbar=m=\omega=1$, $\mathbf r_i=(x_i,y_i)$, and **no interactions**. Store all spin-up coordinates before the spin-down coordinates. All tests should pass with $N_\uparrow=N_\downarrow=1,2,3$.


## 2. Imports, numerical conventions and exact shell occupations

Keep the same plotting and random-generator conventions as the earlier notebook. Configurations have shape `(n_samples, N, 2)`, where the last axis holds `(x,y)`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import factorial
from scipy.special import eval_hermite

rng = np.random.default_rng(1234)
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e1e0d9",
    "axes.axisbelow": True, "lines.linewidth": 2
})
C_VMC, C_EXACT, C_REF = "#2a78d6", "#eb6834", "#898781"
C_FERMI = "#1baf7a"

In [ ]:
def occupied_orbitals(n):
    """Return the lowest n orbitals [(n_x,n_y),...], shell by shell.

    Order within a shell: (s,0), (s-1,1), ..., (0,s).
   """
    labels = []
    s = 0
    while len(labels) < n:
        for nx in range(s, -1, -1):
            labels.append((nx, s-nx))
            if len(labels) == n:
                return labels
        s += 1
    return labels


def exact_energy(n_up, n_down):
    return sum(nx+ny+1 for nx,ny in occupied_orbitals(n_up)) + sum(nx+ny+1 for nx,ny in occupied_orbitals(n_down))


## 3. TODO: implement 2D oscillator values and spatial derivatives

Start with the normalized 1D oscillator orbitals,

$$\varphi_n(x)=\frac{H_n(x)e^{-x^2/2}}{\pi^{1/4}\sqrt{2^n n!}},\quad \phi_{n_xn_y}(x,y)=\varphi_{n_x}(x)\varphi_{n_y}(y).$$

**Derive:** $\partial_x\phi_{n_xn_y}$ and $\partial_y\phi_{n_xn_y}$ using the Hermite recurrence; derive a simple expression for $\nabla^2\phi$ from $h\phi=\epsilon\phi$.

Implement `ho1d`, `orbital_values`, `orbital_values_grad_lap`. Required shapes: `M = (..., n_orbitals)`, `dM = (..., n_orbitals, 2)`, `lapM = (..., n_orbitals)`. The last axis of input `r` is `(x,y)`.

In [ ]:
def ho1d(x, n):
    """Normalized n-th 1D oscillator orbital; scipy.special.eval_hermite is available."""
    return eval_hermite(n, x) * np.exp(-0.5*x**2) / (np.pi**0.25 * np.sqrt(2.0**n * factorial(n)))


def orbital_values(r, labels):
    """Evaluate phi_(nx,ny)(r) for each label; shape r.shape[:-1]+(len(labels),)."""
    x, y = r[..., 0], r[..., 1]
    # Devo fare M_ij = Phi_j (x_i)
    return np.stack([ho1d(x, nx)*ho1d(y, ny) for nx,ny in labels], axis=-1)


def orbital_values_grad_lap(r, labels):
    """Return (M,dM,lapM) with dM[...,orbital,coordinate].
    """
    # TODO: implement first and second coordinate derivatives.
    raise NotImplementedError("TODO: orbital_values_grad_lap")

# CHECKPOINT: your first three orbitals should match these explicit forms.
r_check = rng.normal(size=(12, 2))
M_check = orbital_values(r_check, occupied_orbitals(3))
phi00 = np.exp(-0.5*np.sum(r_check**2,axis=1))/np.sqrt(np.pi)
np.testing.assert_allclose(M_check[:,0], phi00)
np.testing.assert_allclose(M_check[:,1], np.sqrt(2)*r_check[:,0]*phi00)
np.testing.assert_allclose(M_check[:,2], np.sqrt(2)*r_check[:,1]*phi00)
print("Orbital values: elementary checks passed.")

## 4. TODO: spin-resolved Slater determinant and antisymmetry

For each spin, construct a matrix $M^\sigma_{ij}=\phi_j(\mathbf r_i^\sigma)$. Use

$$\Psi=D_\uparrow D_\downarrow,\qquad
\log|\Psi|=\log|D_\uparrow|+\log|D_\downarrow|.$$

**Why `slogdet`?** A fermionic wavefunction has nodes and changes sign; $\log\Psi$ is not globally real. For Metropolis you need $\log|\Psi|$, and for antisymmetry tests you need the sign. Handle empty spin blocks (determinant $1$).

In [ ]:
def slater_sign_logabs(X, n_up):
    """For X shape (samples,N,2), return arrays (sign,logabs) of shape (samples,).

    Hint: X[:,:n_up] and X[:,n_up:] contain the two spin populations.
    Hint: np.linalg.slogdet(M) returns sign and log(abs(det(M))).
    """
    # TODO: construct each Slater matrix and multiply signs / add log amplitudes.
    raise NotImplementedError("TODO: slater_sign_logabs")


def logabs_psi(X, n_up):
    # TODO: expose only the log amplitude to the Metropolis sampler.
    raise NotImplementedError("TODO: logabs_psi")

# CHECKPOINT: one same-spin exchange must flip psi's sign but not |psi|².
X_test = rng.normal(size=(20,6,2))
X_swapped = X_test.copy()
X_swapped[:,[0,1],:] = X_test[:,[1,0],:]
s1,l1 = slater_sign_logabs(X_test, 3)
s2,l2 = slater_sign_logabs(X_swapped, 3)
np.testing.assert_array_equal(s2, -s1)
np.testing.assert_allclose(l2, l1, atol=1e-12)
print("Same-spin exchange: sign flips and |psi|² is unchanged.")

## 5. TODO: adapt the Metropolis sampler from 1D to 2D

Use your previous `metropolis_step` and `sample_mcmc` structure. A proposed step should move **one selected particle** by a vector $\boldsymbol\delta\in[-\Delta,\Delta]^2$. A sweep consists of $N$ such attempts. Sampling depends only on $\log|\Psi|$, not its sign.

$$A(X\to X')=\min\left(1,e^{2[\log|\Psi(X')|-\log|\Psi(X)|]}\right).$$

**Important shape change:** one configuration `(N,2)`, samples `(n_samples,N,2)`.

In [ ]:
def metropolis_step(logpsi, x, logpsi_x, step_size, rng=rng):
    """One Metropolis update for x.shape == (N,2)."""
    # TODO: evaluate logpsi_x if None; choose a particle; copy x; propose
    # a 2D displacement; compute log acceptance using 2*(new-old);
    # return (new_or_old_x, new_or_old_logpsi, accepted_boolean).
    raise NotImplementedError("TODO: metropolis_step")


def sample_mcmc(logpsi, n_particles, n_samples, n_discard, step_size,
                x0=None, rng=rng, return_acceptance=False):
    """One sample after each sweep; shape (n_samples,N,2)."""
    # TODO: adapt your previous sweep loop; x0 shape (N,2).
    raise NotImplementedError("TODO: sample_mcmc")

# CHECKPOINT:
smoke, acceptance = sample_mcmc(lambda X: logabs_psi(X,2),
                                4, n_samples=250, n_discard=100,
                                step_size=1.0, return_acceptance=True)
assert smoke.shape == (250,4,2)
print("2+2 sampler: shape", smoke.shape, "acceptance", round(acceptance,3))

## 6. TODO: analytic local energy from the orbital derivatives

For $L(X)=\log|\Psi(X)|$, away from nodes,

$$E_L(X)=-\frac12\sum_i\big[\nabla_i^2L+|\nabla_iL|^2\big]+\frac12\sum_i r_i^2.$$

**Derive** the determinant gradient using $\partial\log|\det M|=\operatorname{tr}(M^{-1}\partial M)$, then note that only row $i$ depends on particle $i$:

$$\nabla_iL=\sum_j(M^{-1})_{ji}\nabla\phi_j(\mathbf r_i).$$

Show similarly that $\nabla_i^2L=(\nabla_i^2D)/D-|\nabla_iL|^2$. Implement both the log-derivative local energy and the **independent** Laplacian-matrix expression; they should agree off the nodes. Use `np.linalg.solve`, not a manually formed matrix inverse, whenever possible.

In [ ]:
def grad_lap_logabs(X, n_up):
    """Return grad shape (samples,N,2), lap_logabs shape (samples,N).

    Hint: build M, dM, lapM for each spin; use (M^-1)_(j,i)
    contracted with derivatives in row i; subtract |grad logabs|^2
    in the per-particle Laplacian.
    """
    # TODO: compute both spin blocks and concatenate results into original particle order.
    raise NotImplementedError("TODO: grad_lap_logabs")


def local_energy_from_log_derivs(X, n_up):
    """Use -1/2 sum_i(lap_i logabs + |grad_i logabs|^2)+sum_i r_i²/2."""
    # TODO: sum over both particles and spatial coordinates correctly.
    raise NotImplementedError("TODO: local_energy_from_log_derivs")


def local_energy_matrix(X, n_up):
    """Alternative kinetic expression: -1/2 trace(M^-1 lapM), spin by spin."""
    # TODO: compute a numerically stable solve and trace in each spin sector.
    raise NotImplementedError("TODO: local_energy_matrix")

# CHECKPOINT: both expressions must reproduce exact E0=10 for any X off the nodes.
X_check = rng.normal(size=(25,6,2))
e_log = local_energy_from_log_derivs(X_check,3)
e_mat = local_energy_matrix(X_check,3)
np.testing.assert_allclose(e_log,e_mat,atol=1e-10,rtol=1e-12)
np.testing.assert_allclose(e_mat,10.,atol=1e-10)
print("Energy, N=6: exact",exact_energy(3,3),
      "max |E_log-E_matrix| =",np.max(np.abs(e_log-e_mat)))

## 7. TODO: automatic differentiation of the determinant (JAX)

The assignment asks for **AD first / analytic orbital derivative comparison**. The previous section gives the analytic path; here implement the **independent AD computation** using `jax.grad` and `jax.hessian` on $L(X)=\log|D_\uparrow D_\downarrow|$.

For one configuration `X.shape=(N,2)`, the Hessian has shape `(N,2,N,2)` and its full coordinate trace is `jnp.einsum('iaia->', hessian)`. Enable float64. **Do not** call NumPy / SciPy functions inside the differentiated function: use `jax.numpy` and an elementary Hermite recurrence.

If JAX is not available, install it in your environment before completing this benchmark; all other sections work with NumPy/SciPy.

In [ ]:
try:
    import jax
    import jax.numpy as jnp
    jax.config.update('jax_enable_x64',True)
    HAS_JAX = True
except ImportError:
    HAS_JAX = False
    print('JAX unavailable: install jax to run this independent check.')

if HAS_JAX:
    def ho1d_jax(x,n):
        """JAX-compatible normalized Hermite orbital. Hint: H_(n+1)=2*x*H_n-2*n*H_(n-1)."""
        # TODO: implement the Hermite recurrence using jnp operations.
        raise NotImplementedError("TODO: ho1d_jax")

    def logabs_single_jax(X,n_up):
        """Scalar logabs for one configuration X.shape=(N,2), using jnp.linalg.slogdet."""
        # TODO: build each spin matrix from JAX orbitals, return sum of logabs.
        raise NotImplementedError("TODO: logabs_single_jax")

    def energy_ad_single(X,n_up):
        """Kinetic energy from grad/hessian of determinant logabs plus harmonic potential."""
        # TODO: jax.grad, jax.hessian and trace jnp.einsum('iaia->',H).
        raise NotImplementedError("TODO: energy_ad_single")

    # CHECKPOINT: compare AD gradient, Laplacian, and energy against Section 6.
    X_ad = rng.normal(size=(1,4,2))
    f = lambda Z: logabs_single_jax(Z,2)
    g_ad = np.asarray(jax.grad(f)(jnp.asarray(X_ad[0])))
    h_ad = np.asarray(jax.hessian(f)(jnp.asarray(X_ad[0])))
    g_analytic,lap_analytic = grad_lap_logabs(X_ad,2)
    np.testing.assert_allclose(g_ad,g_analytic[0],atol=1e-10)
    np.testing.assert_allclose(np.einsum('iaia->',h_ad),np.sum(lap_analytic[0]),atol=1e-10)
    e_ad = float(energy_ad_single(jnp.asarray(X_ad[0]),2))
    np.testing.assert_allclose(e_ad,local_energy_matrix(X_ad,2)[0],atol=1e-10)
    print(f'AD check: E_AD={e_ad:.12f}, E_exact={exact_energy(2,2)}; first/second derivatives agree.')

## 8. TODO: VMC energy benchmarks ($N=2,4,6$)

Reuse the blocking estimator from the old notebook. Simulate $1+1$, $2+2$, $3+3$ using exact oscillator orbitals. Compare to the shell-filling answer. **Explain why the exact local-energy variance must vanish** and why the reported MC error becomes dominated by roundoff rather than statistical noise.

No variational optimization is needed for this stage: you are using exact one-body orbitals at $\omega=1$.

In [ ]:
def estimate_energy(eloc, n_blocks=20):
    """Mean and blocking standard error: reuse previous notebook's logic."""
    # TODO: form contiguous block means and estimate SEM between them.
    raise NotImplementedError("TODO: estimate_energy")


def vmc_fermions(n_up,n_down,n_samples=2500,n_discard=400,step_size=1.1):
    """Return X, eloc, mean energy, error, acceptance fraction."""
    # TODO: combine Sections 4, 5, and 6 into a complete VMC run.
    raise NotImplementedError("TODO: vmc_fermions")

runs = {}
for nu,nd in [(1,1),(2,2),(3,3)]:
    X,eloc,e,err,accept = vmc_fermions(nu,nd,
                                         n_samples=5000 if nu==3 else 2500)
    runs[(nu,nd)] = X
    expected = exact_energy(nu,nd)
    print(f'{nu}+{nd}: VMC E={e:.12f} ± {err:.2g} | exact={expected} | '
          f'acceptance={accept:.3f} | max local deviation={np.max(np.abs(eloc-expected)):.2g}')
    assert np.abs(e-expected)<1e-10

fig,ax = plt.subplots(figsize=(6.6,3.7))
for (nu,nd),X in runs.items():
    e=local_energy_matrix(X,nu)
    ax.plot(np.arange(min(250,len(e))),e[:250],label=f'{nu}+{nd}: E={exact_energy(nu,nd)}',lw=1.4)
ax.set(xlabel='Recorded sweep',ylabel=r'Local energy $E_L$',title='Exact-eigenstate VMC: zero-variance benchmark')
ax.legend(frameon=False)
plt.show()

## 9. TODO: radial number density with normalization $N$

Define $n(\mathbf r)=\langle\sum_i\delta^{(2)}(\mathbf r-\mathbf r_i)\rangle$ and its angular average $n(r)$. Your convention **must** satisfy

$$\boxed{2\pi\int_0^\infty r\,n(r)\,dr=N.}$$

Use bins corresponding to **annuli** of area $\pi(r_{k+1}^2-r_k^2)$; a plain radial histogram with `density=True` gives a *radial probability density*, not the spatial number density $n(r)$. Construct the exact benchmark from the occupied orbitals and average it over angle (important for the $2+2$ open shell).

**Checkpoint:** $n_{3+3}(r)$ can be obtained analytically by summing $|\phi_{00}|^2+|\phi_{10}|^2+|\phi_{01}|^2$ for both spins.

In [ ]:
def radial_number_density(samples,bins):
    """Return (annulus centers, number density, edges), normalized to N."""
    # TODO: sample radii; count annulus entries; divide by sample count and annulus area.
    raise NotImplementedError("TODO: radial_number_density")


def exact_radial_density(r,n_up,n_down,n_angles=96):
    """Angularly average sum of |occupied orbital|² for both spins."""
    # TODO: evaluate the spatial density on a polar mesh; average over theta.
    raise NotImplementedError("TODO: exact_radial_density")

bins = np.linspace(0,5.5,60)
fig,axes=plt.subplots(1,3,figsize=(13,3.6),sharey=True)
for ax,(nu,nd) in zip(axes,[(1,1),(2,2),(3,3)]):
    X = runs[(nu,nd)]
    r,density,edges = radial_number_density(X,bins)
    r_fine = np.linspace(0,5.5,500)
    ax.plot(r_fine,exact_radial_density(r_fine,nu,nd),color=C_EXACT,label='Exact orbital density')
    ax.plot(r,density,'.',color=C_VMC,label='VMC annular density',markersize=4)
    integrated = np.sum(density*np.pi*(edges[1:]**2-edges[:-1]**2))
    ax.set(xlabel=r'$r$',title=f'{nu}+{nd}: annular integral = {integrated:.3f}')
    assert abs(integrated-(nu+nd)) < 0.05
axes[0].set_ylabel(r'$n(r)$ (particles per unit area)')
axes[0].legend(frameon=False,fontsize=9)
fig.tight_layout()
plt.show()

r_test = np.linspace(0,3,30)
np.testing.assert_allclose(exact_radial_density(r_test,3,3),
    (2/np.pi)*np.exp(-r_test**2)*(1+2*r_test**2),atol=1e-12)

## 10. TODO: same-spin pair distribution and the exchange hole

Use **only same-spin unordered pairs** and Euclidean distances in 2D. If

$$C=\binom{N_\uparrow}{2}+\binom{N_\downarrow}{2}>0,$$

establish your convention

$$P_\mathrm{same}(d)=\frac1C\left\langle\sum_{\sigma}\sum_{i<j\in\sigma}\delta(d-|\mathbf r_i-\mathbf r_j|)\right\rangle,\qquad
\boxed{\int_0^\infty P_\mathrm{same}(d)\,dd=1.}$$

**Warning:** $P(d)$ for *any* smooth 2D pair distribution approaches zero at the origin because the area of a thin annulus scales as $2\pi d\,dd$. To expose the exchange hole, compare $P(d)/(2\pi d)$ against a reference where each particle column is shuffled **independently across configurations** (preserves one-body density, destroys pair correlation). At $1+1$ no same-spin pairs exist: return an empty array / mark the observable undefined.

In [ ]:
def same_spin_pair_distances(samples,n_up):
    """Return shape (n_samples,C) with only same-spin unordered 2D distances."""
    # TODO: use np.triu_indices for each spin block and np.linalg.norm(...,axis=-1).
    # Return shape (n_samples,0) when C=0.
    raise NotImplementedError("TODO: same_spin_pair_distances")


def independent_reference_samples(samples,rng=rng):
    """Break within-sample correlations while preserving each particle marginal."""
    # TODO: independently permute sample indices for each particle column.
    raise NotImplementedError("TODO: independent_reference_samples")

X = runs[(3,3)]
d_real = same_spin_pair_distances(X,3).ravel()
d_reference = same_spin_pair_distances(independent_reference_samples(X),3).ravel()
assert d_real.size == len(X)*6
edges=np.linspace(0,max(d_real.max(),d_reference.max())+1e-6,62)
counts,_=np.histogram(d_real,bins=edges)
counts_ref,_=np.histogram(d_reference,bins=edges)
centers=0.5*(edges[1:]+edges[:-1])
widths=np.diff(edges)
P=counts/(len(X)*6*widths)
P_ref=counts_ref/(len(X)*6*widths)
assert np.isclose(np.sum(P*widths),1.)
assert np.isclose(np.sum(P_ref*widths),1.)

fig,(ax_pair,ax_area)=plt.subplots(1,2,figsize=(11,4))
ax_pair.plot(centers,P,color=C_FERMI,label='Fermions (same spin)')
ax_pair.plot(centers,P_ref,color=C_REF,label='Independent-particle reference')
ax_pair.set(xlabel=r'Pair separation $d$',ylabel=r'$P_{\mathrm{same}}(d)$',
            title='Pair-distance PDF (integral = 1)',xlim=(0,3.6))
ax_pair.legend(frameon=False)
ax_area.plot(centers,P/(2*np.pi*centers),color=C_FERMI,label='Fermions')
ax_area.plot(centers,P_ref/(2*np.pi*centers),color=C_REF,label='Independent reference')
ax_area.set(xlabel=r'Pair separation $d$',ylabel=r'$P_{\mathrm{same}}(d)/(2\pi d)$',
            title='Area-normalized pair density: exchange hole',xlim=(0,2.1))
ax_area.legend(frameon=False)
fig.tight_layout()
plt.show()

## 11. Checklist / questions to submit with your results

- [ ] Derive the 2D oscillator spectrum, degeneracies, and exact shell-filling energies.
- [ ] Construct both spin Slater matrices and show antisymmetry under **one** same-spin swap.
- [ ] Explain why **two** simultaneous same-spin swaps give a plus sign.
- [ ] Show particle-by-particle Metropolis sampling using $\log|\Psi|$ and 2D moves.
- [ ] Cross-check automatic differentiation against analytic orbital derivatives.
- [ ] Show $E_0=2,6,10$ numerically; explain the zero-variance property.
- [ ] State $2\pi\int_0^\infty r n(r)dr=N$ and compare radial densities with exact orbitals.
- [ ] State $\int_0^\infty P_\mathrm{same}(d)dd=1$; explain why a 2D exchange hole is best displayed after dividing by $2\pi d$ or comparing to a decorrelated reference.

**Short-answer questions:**

1. Why can a product $D_\uparrow D_\downarrow$ represent fermions even though a *full* spin-space state is one determinant?
2. Why is $N=4$ degenerate and how does the choice of $\phi_{10}$ or $\phi_{01}$ affect the 2D density?
3. Why is the local-energy variance zero for the exact Slater ansatz?
4. What would need changing to add a symmetric Jastrow correlation factor later?

**Reading:** Bayha et al., *Nature* **587**, 583–587 (2020), DOI `10.1038/s41586-020-2936-y`; Foulkes et al., *Rev. Mod. Phys.* **73**, 33 (2001), Slater determinants and VMC.